In [1]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)
    
# os.chdir(module_path)
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /home/fre.gilad/source/AgentDaC/notebooks


## Env

In [2]:
from src.utils.env import prepare_environment

prepare_environment("../api_keys")

INFO 08-18 18:27:10 [src.utils.env:30] Setting API tokens: ['WANDB_API_KEY', 'OPENPIPE_API_KEY', 'OPENAI_API_KEY']
INFO 08-18 18:27:10 [src.utils.env:42] Setting additional variables: {'NCCL_CUMEM_ENABLE': '0', 'VLLM_USE_V1': '0', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn', 'ART_SERVER_TIMEOUT': '300'}


## Model

In [3]:
from src.configs import art_configs
from src.configs import vllm_configs
from src.models import PathConfig

print("Available ART model configurations:")
print(art_configs.available_configs())

print("Available vLLM model configurations:")
print(vllm_configs.available_configs())

base_model = "unsloth/Qwen3-14B"
project_name = "easy2hard_dac"

path_config = PathConfig(
    base_model=base_model,
    project_name=project_name,
)

Available ART model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B']
Available vLLM model configurations:
['unsloth/Llama-4-Scout-17B-16E-Instruct', 'unsloth/Llama-4-Scout-17B-16E-Instruct-unsloth-dynamic-bnb-4bit', 'unsloth/Qwen2-7B', 'unsloth/Qwen2.5-14B-Instruct', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit', 'unsloth/Qwen2.5-32B-Instruct', 'unsloth/Qwen3-14B', 'unsloth/Qwen3-32B', 'unsloth/Qwen3-32B-unsloth-bnb-4bit']


In [4]:
import art

host = "0.0.0.0"
port = 8200

model = art.Model(
    name=base_model,
    project=path_config.project_name,
    inference_api_key=os.getenv("OPENAI_API_KEY", "default"),
    inference_base_url=f"https://{host}:{port}/v1",
)

## Data

In [5]:
from datasets import Dataset, load_dataset, DatasetDict

dataset_dict: DatasetDict = load_dataset(
    path="furonghuang-lab/Easy2Hard-Bench",
    name="E2H-AMC",
    split=None,
)  # type: ignore

train_data: Dataset = dataset_dict["train"]
test_data: Dataset = dataset_dict["eval"]

## Inference Clients

In [6]:
from src.vllm_client import VllmClient, VllmRouter

inference_clients: list[VllmClient] = []
vllm_server_ports = [8200]
for port in vllm_server_ports:
    inference_clients.append(VllmClient.from_connection(port=port, base_model=base_model))

vllm_router: VllmRouter = VllmRouter(inference_clients)

## Config 

In [7]:
from experiments.easy2hard.trainer import Easy2HardTrainer
from src.trainer import PromptConfig, StopCriteria
from src.dac_agent_tools import AgentToolNode

prompt_config = PromptConfig(
    system_root="",
    system_inter="",
    system_leaf="",
)

stop_criteria = StopCriteria(max_depth=1)

class Easy2HardToolsTrainer(Easy2HardTrainer):
    def create_agent(self) -> AgentToolNode:
        client = self.vllm_router.next()
        return AgentToolNode(
            model_name=self.model.get_inference_name(),
            openai_client=client.openai_client,
            prompt_config=self.prompt_config,
            stop_criteria=self.stop_criteria,
        )
    

trainer = Easy2HardToolsTrainer(
    model=model,
    vllm_router=vllm_router,
    path_config=path_config,
    prompt_config=prompt_config,
    stop_criteria=stop_criteria,
)

### Inference Test

In [8]:
import random

idx = random.randint(0, len(train_data) - 1)
# idx = 228
print(f"Selected random index: {idx}")
sample = train_data[idx]
problem = sample["problem"]
true_answer = sample["answer"]

print(f"Problem: {problem}")
print(f"Answer: {true_answer}")
print("-" * 200)
print()

preds = await trainer.predict([sample], verbose=True, seed=0, extra_body={"chat_template_kwargs": {"enable_thinking": False}})

Selected random index: 766
Problem: Let $N$ be the positive integer $7777\ldots777$, a $313$-digit number where each digit is a $7$. Let $f(r)$ be the leading digit of the $r{ }$th root of $N$. What is\[f(2) + f(3) + f(4) + f(5)+ f(6)?\]
Answer: 8
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------



predict:   0%|          | 0/1 [00:00<?, ?it/s]

Role:
USER
Content:
Let $N$ be the positive integer $7777\ldots777$, a $313$-digit number where each digit is a $7$. Let $f(r)$ be the leading digit of the $r{ }$th root of $N$. What is\[f(2) + f(3) + f(4) + f(5)+ f(6)?\]



Role:
ASSISTANT
Content:
<tool_call>
{"name": "delegate_sub_task", "arguments": {"input": "Let $N$ be the positive integer $7777\ldots777$, a $313$-digit number where each digit is a $7$. Let $f(r)$ be the leading digit of the $r$th root of $N$. What is$$f(2) + f(3) + f(4) + f(5)+ f(6)?$$"}}
</tool_call>

